# lltMemoryThroughput part 2 measurements (2x T4)

Runs allreduce.py, ddp.py, fsdp.py and fsdp_ckpt.py exactly as shipped with torchrun and records their output.

In [ ]:

import io, json, os, sys, time, contextlib, platform, subprocess
RUN = {"module": "lltMemoryThroughput", "started": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "stdout": {}, "data": {}}

class Tee(io.StringIO):
    def write(self, s):
        sys.__stdout__.write(s)
        return super().write(s)

def run_step(key, code):
    buf = Tee()
    t = time.time()
    with contextlib.redirect_stdout(buf):
        exec(code, globals())
    RUN["stdout"][key] = buf.getvalue()
    RUN.setdefault("stepSec", {})[key] = round(time.time() - t, 1)
    print(buf.getvalue())

def save(name):
    import torch
    RUN["env"] = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
                  "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
    RUN["finished"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    with open(f"/kaggle/working/{name}.json", "w") as f:
        json.dump(RUN, f, indent=1, default=str)
    print("saved", name)

def torchrun(key, args, timeout=1800):
    cmd = [sys.executable, "-m", "torch.distributed.run"] + args
    t = time.time()
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    out = p.stdout + ("" if p.returncode == 0 else "\nSTDERR TAIL\n" + p.stderr[-4000:])
    RUN["stdout"][key] = out
    RUN.setdefault("stepSec", {})[key] = round(time.time() - t, 1)
    RUN.setdefault("returncode", {})[key] = p.returncode
    print(key, "rc", p.returncode, round(time.time() - t, 1), "s")
    print(out)

import torch
print(torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print(subprocess.run(['nvidia-smi', 'topo', '-m'], capture_output=True, text=True).stdout)
RUN['data']['topo'] = subprocess.run(['nvidia-smi', 'topo', '-m'], capture_output=True, text=True).stdout


In [ ]:
with open('gpt.py', 'w') as f:
    f.write('# gpt.py - GPT-2 small (124M) in plain PyTorch, with the two memory switches this lab measures.\nimport math\nfrom dataclasses import dataclass\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n\n@dataclass\nclass GPTConfig:\n    vocab: int = 50257\n    ctx: int = 1024\n    layers: int = 12\n    heads: int = 12\n    d: int = 768\n\n\nclass Block(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.heads = c.heads\n        self.ln1, self.ln2 = nn.LayerNorm(c.d), nn.LayerNorm(c.d)\n        self.qkv, self.proj = nn.Linear(c.d, 3 * c.d), nn.Linear(c.d, c.d)\n        self.fc, self.out = nn.Linear(c.d, 4 * c.d), nn.Linear(4 * c.d, c.d)\n\n    def forward(self, x):\n        B, T, C = x.shape\n        q, k, v = self.qkv(self.ln1(x)).split(C, dim=2)\n        q, k, v = (t.view(B, T, self.heads, -1).transpose(1, 2) for t in (q, k, v))\n        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)\n        x = x + self.proj(y.transpose(1, 2).reshape(B, T, C))\n        return x + self.out(F.gelu(self.fc(self.ln2(x)), approximate="tanh"))\n\n\ndef chunk_loss(h, w, y):\n    return F.cross_entropy((h @ w.t()).float(), y, reduction="sum")\n\n\nclass GPT(nn.Module):\n    def __init__(self, c, ckpt=False, chunks=0):\n        super().__init__()\n        self.ckpt, self.chunks = ckpt, chunks   # activation checkpointing, chunked cross-entropy\n        self.wte, self.wpe = nn.Embedding(c.vocab, c.d), nn.Embedding(c.ctx, c.d)\n        self.blocks = nn.ModuleList([Block(c) for _ in range(c.layers)])\n        self.ln_f = nn.LayerNorm(c.d)\n        self.head = nn.Linear(c.d, c.vocab, bias=False)\n        self.head.weight = self.wte.weight          # tied, as in GPT-2\n        for n, p in self.named_parameters():\n            if p.dim() == 2:\n                small = n.endswith(("proj.weight", "out.weight"))\n                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * c.layers) if small else 0.02)\n            elif n.endswith("bias"):\n                nn.init.zeros_(p)\n\n    def forward(self, idx, targets):\n        x = self.wte(idx) + self.wpe(torch.arange(idx.shape[1], device=idx.device))\n        for b in self.blocks:\n            x = checkpoint(b, x, use_reentrant=False) if self.ckpt else b(x)\n        h, y = self.ln_f(x).flatten(0, 1), targets.flatten()\n        if not self.chunks:\n            return F.cross_entropy(self.head(h).float(), y)\n        # Chunked cross-entropy: each chunk\'s logits are recomputed in backward, so one chunk is alive at a time\n        parts = [checkpoint(chunk_loss, hc, self.head.weight, yc, use_reentrant=False)\n                 for hc, yc in zip(h.chunk(self.chunks), y.chunk(self.chunks))]\n        return sum(parts) / y.numel()\n')
print('wrote', 'gpt.py')


In [ ]:
with open('allreduce.py', 'w') as f:
    f.write('# allreduce.py - launch with: torchrun --nproc_per_node=2 allreduce.py\nimport time\nimport torch\nimport torch.distributed as dist\n\ndist.init_process_group("nccl")\nrank, n = dist.get_rank(), dist.get_world_size()\ntorch.cuda.set_device(rank)\nfor mib in (1, 16, 64, 256, 512):\n    x = torch.ones(mib * 2**20 // 4, device="cuda")      # mib MiB of fp32\n    for _ in range(3):\n        dist.all_reduce(x)                                # warm-up, sets up the NCCL channels\n    torch.cuda.synchronize()\n    t = time.perf_counter()\n    for _ in range(10):\n        dist.all_reduce(x)\n    torch.cuda.synchronize()\n    dt = (time.perf_counter() - t) / 10\n    algbw = x.numel() * 4 / dt / 1e9                      # message bytes per second\n    busbw = algbw * 2 * (n - 1) / n                       # bytes each GPU actually sent per second\n    if rank == 0:\n        print(f"{mib:>4} MiB  {1e3 * dt:7.2f} ms  algbw {algbw:5.2f} GB/s  busbw {busbw:5.2f} GB/s")\ndist.destroy_process_group()\n')
print('wrote', 'allreduce.py')


In [ ]:
with open('ddp.py', 'w') as f:
    f.write('# ddp.py - launch with: torchrun --nproc_per_node=2 ddp.py [--sync-every-micro] [--same-seed]\nimport contextlib, sys, time\nimport torch\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom gpt import GPT, GPTConfig\n\ndist.init_process_group("nccl")\nrank, world = dist.get_rank(), dist.get_world_size()\ntorch.cuda.set_device(rank)\ntorch.manual_seed(0)                                   # same init everywhere; DDP also broadcasts rank 0\'s weights\nmodel = DDP(GPT(GPTConfig()).cuda(), device_ids=[rank])\nopt = torch.optim.AdamW(model.parameters(), lr=6e-4, fused=True)\nscaler = torch.amp.GradScaler("cuda")\ndata_seed = 1234 if "--same-seed" in sys.argv else 1234 + rank     # each rank must draw different data\ngen = torch.Generator(device="cuda").manual_seed(data_seed)\nB, S, ACCUM, STEPS = 8, 1024, 4, 10\n\nfor step in range(STEPS + 3):\n    if step == 3:                                      # 3 warm-up steps, then start the clock\n        torch.cuda.synchronize()\n        t = time.perf_counter()\n    for i in range(ACCUM):\n        x = torch.randint(0, 50257, (B, S), device="cuda", generator=gen)\n        last = i == ACCUM - 1\n        # no_sync skips the gradient all-reduce; only the last micro-batch communicates\n        ctx = contextlib.nullcontext() if last or "--sync-every-micro" in sys.argv else model.no_sync()\n        with ctx:\n            with torch.autocast("cuda", dtype=torch.float16):\n                loss = model(x, x) / ACCUM\n            scaler.scale(loss).backward()\n    scaler.step(opt)\n    scaler.update()\n    opt.zero_grad(set_to_none=True)\ntorch.cuda.synchronize()\ntps = STEPS * ACCUM * B * S * world / (time.perf_counter() - t)\nlosses = [None] * world\ndist.all_gather_object(losses, round(loss.item() * ACCUM, 6))\nif rank == 0:\n    print(f"GPUs {world}  {\' \'.join(sys.argv[1:]) or \'default\'}  {tps:,.0f} tokens/s  last loss per rank {losses}")\ndist.destroy_process_group()\n')
print('wrote', 'ddp.py')


In [ ]:
with open('fsdp.py', 'w') as f:
    f.write('# fsdp.py - launch with: torchrun --nproc_per_node=2 fsdp.py  (or 1, to see one T4 run out of memory)\nimport time\nimport torch\nimport torch.distributed as dist\nfrom torch.distributed.fsdp import fully_shard, MixedPrecisionPolicy\nfrom gpt import GPT, GPTConfig\n\nCFG = GPTConfig(layers=24, heads=16, d=2048)          # about 1.3B parameters\nGB = 1e9\n\ndef build():\n    torch.manual_seed(0)\n    model = GPT(CFG, ckpt=True, chunks=8)              # built full size on the CPU\n    mp = MixedPrecisionPolicy(param_dtype=torch.float16, reduce_dtype=torch.float32)\n    for block in model.blocks:\n        fully_shard(block, mp_policy=mp)               # each block is gathered, used and freed as one unit\n    fully_shard(model, mp_policy=mp)                   # embeddings, final norm and the tied head\n    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)\n    return model, opt\n\ndef train(model, opt, scaler, steps, gen, B=2, S=1024):\n    for _ in range(steps):\n        x = torch.randint(0, CFG.vocab, (B, S), device="cuda", generator=gen)\n        loss = model(x, x)\n        scaler.scale(loss).backward()\n        scaler.step(opt)\n        scaler.update()\n        opt.zero_grad(set_to_none=True)\n    return loss.item()\n\nif __name__ == "__main__":\n    dist.init_process_group("nccl")\n    rank, world = dist.get_rank(), dist.get_world_size()\n    torch.cuda.set_device(rank)\n    model, opt = build()\n    N = sum(p.numel() for p in model.parameters())\n    scaler = torch.amp.GradScaler("cuda")\n    gen = torch.Generator(device="cuda").manual_seed(100 + rank)\n    try:\n        train(model, opt, scaler, 2, gen)\n        torch.cuda.synchronize()\n        torch.cuda.reset_peak_memory_stats()\n        t = time.perf_counter()\n        loss = train(model, opt, scaler, 5, gen)\n        torch.cuda.synchronize()\n        tps = 5 * 2 * 1024 * world / (time.perf_counter() - t)\n        print(f"rank {rank} of {world}  N {N:,}  peak {torch.cuda.max_memory_allocated() / GB:.2f} GB  "\n              f"16N/{world} = {16 * N / world / GB:.2f} GB  {tps:,.0f} tokens/s  loss {loss:.3f}")\n    except torch.cuda.OutOfMemoryError:\n        print(f"rank {rank} of {world}  N {N:,}  OUT OF MEMORY at {torch.cuda.memory_allocated() / GB:.2f} GB allocated, "\n              f"16N = {16 * N / GB:.2f} GB")\n    dist.destroy_process_group()\n')
print('wrote', 'fsdp.py')


In [ ]:
with open('fsdp_ckpt.py', 'w') as f:
    f.write('# fsdp_ckpt.py - launch with: torchrun --nproc_per_node=2 fsdp_ckpt.py\nimport gc, os, time\nimport torch\nimport torch.distributed as dist\nimport torch.distributed.checkpoint as dcp\nfrom torch.distributed.checkpoint.state_dict import get_state_dict, set_state_dict\nfrom fsdp import build, train, CFG\nfrom gpt import GPT\n\ndist.init_process_group("nccl")\nrank = dist.get_rank()\ntorch.cuda.set_device(rank)\nPATH = "/tmp/ckpt_fsdp"\n\nmodel, opt = build()\nscaler = torch.amp.GradScaler("cuda")\ntrain(model, opt, scaler, 3, torch.Generator(device="cuda").manual_seed(100 + rank))\n\n# Wrong. Rank 0 alone saves its state_dict. Under FSDP every tensor in it holds only rank 0\'s shard.\nif rank == 0:\n    torch.save(model.state_dict(), "/tmp/rank0_only.pt")\n\n# Right. Every rank writes its own shards, model and optimizer, plus the loss scaler.\nt = time.perf_counter()\nmsd, osd = get_state_dict(model, opt)\ndcp.save({"model": msd, "optim": osd, "scaler": scaler.state_dict()}, checkpoint_id=PATH)\ndist.barrier()\nsave_s = time.perf_counter() - t\nloss_a = train(model, opt, scaler, 2, torch.Generator(device="cuda").manual_seed(7 + rank))\ndel model, opt, msd, osd\ngc.collect()\ntorch.cuda.empty_cache()\n\n# A fresh model and optimizer load the checkpoint, then take the same 2 steps on the same batches.\nmodel, opt = build()\nt = time.perf_counter()\nmsd, osd = get_state_dict(model, opt)\nstate = {"model": msd, "optim": osd, "scaler": scaler.state_dict()}\ndcp.load(state, checkpoint_id=PATH)\nset_state_dict(model, opt, model_state_dict=state["model"], optim_state_dict=state["optim"])\nscaler.load_state_dict(state["scaler"])\nload_s = time.perf_counter() - t\nloss_b = train(model, opt, scaler, 2, torch.Generator(device="cuda").manual_seed(7 + rank))\n\nif rank == 0:\n    files = {f: round(os.path.getsize(os.path.join(PATH, f)) / 1e9, 2) for f in sorted(os.listdir(PATH))}\n    print("sharded checkpoint files (GB):", files)\n    print(f"save {save_s:.1f} s  load {load_s:.1f} s  loss after 2 more steps: continued {loss_a:.6f}  reloaded {loss_b:.6f}")\n    sd = torch.load("/tmp/rank0_only.pt", weights_only=False)\n    kept = sum(v.to_local().numel() for v in sd.values())\n    print(f"rank-0-only file {os.path.getsize(\'/tmp/rank0_only.pt\') / 1e9:.2f} GB, holds {kept:,} values for a model of "\n          f"{sum(v.numel() for v in sd.values()):,}")\n    try:\n        GPT(CFG).load_state_dict({k: v.to_local() for k, v in sd.items()})\n    except RuntimeError as e:\n        first = [line.strip() for line in str(e).splitlines() if "size mismatch" in line][:1]\n        print("loading it into a full model:", first or str(e)[:300])\ndist.destroy_process_group()\n')
print('wrote', 'fsdp_ckpt.py')


In [ ]:
torchrun('allreduce', ['--nproc_per_node=2', 'allreduce.py'])
save('lltMemoryThroughput_p2')


In [ ]:
torchrun('ddp_1gpu', ['--nproc_per_node=1', 'ddp.py'])
torchrun('ddp_2gpu', ['--nproc_per_node=2', 'ddp.py'])
torchrun('ddp_2gpu_sync_every_micro', ['--nproc_per_node=2', 'ddp.py', '--sync-every-micro'])
torchrun('ddp_2gpu_same_seed', ['--nproc_per_node=2', 'ddp.py', '--same-seed'])
save('lltMemoryThroughput_p2')


In [ ]:
torchrun('fsdp_1gpu', ['--nproc_per_node=1', 'fsdp.py'])
torchrun('fsdp_2gpu', ['--nproc_per_node=2', 'fsdp.py'])
save('lltMemoryThroughput_p2')


In [ ]:
torchrun('fsdp_ckpt', ['--nproc_per_node=2', 'fsdp_ckpt.py'])
import shutil
shutil.rmtree('/tmp/ckpt_fsdp', ignore_errors=True)
save('lltMemoryThroughput_p2')
